# الفصل 48: تقنيات RAG المتقدمة وGraphRAG

> RAG الأساسي (Naive RAG) يفشل في كثير من السيناريوهات الواقعية: أسئلة تتطلب ربط معلومات من مصادر متعددة، أسئلة غامضة الصياغة، أو حاجة لفهم العلاقات بين الكيانات.

- **الجزء:** 10 — التطبيقات والوكلاء الذكية
- **الفصل:** 48
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 47: أسس التوليد المعزّز بالاسترجاع: RAG
- **الوسوم:** Re-ranking، Hybrid Search، GraphRAG، Agentic RAG

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch48-advanced-rag.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** networkx, rank-bm25, sentence-transformers
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("networkx", "networkx"),
    ("rank_bm25", "rank-bm25"),
    ("sentence_transformers", "sentence-transformers")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch48-advanced-rag.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch48-advanced-rag.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> RAG الأساسي (Naive RAG) يفشل في كثير من السيناريوهات الواقعية: أسئلة تتطلب ربط معلومات من مصادر متعددة، أسئلة غامضة الصياغة، أو حاجة لفهم العلاقات بين الكيانات.

> **بيانات الفصل**
>
> - **الجزء:** 10 — التطبيقات والوكلاء الذكية
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 47: أسس التوليد المعزّز بالاسترجاع: RAG


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم إعادة الترتيب (Re-ranking) — تحسين سريع وفعّال جدًا
- فهم البحث الهجين (Hybrid Search): دلالي + كلمات مفتاحية
- فهم RAG الوكيلي (Agentic RAG) — تمهيد للفصل القادم


## 1. لماذا يهمّك هذا الموضوع؟

RAG الأساسي (Naive RAG) يفشل في كثير من السيناريوهات الواقعية: أسئلة تتطلب ربط معلومات من مصادر متعددة، أسئلة غامضة الصياغة، أو حاجة لفهم العلاقات بين الكيانات. هذا الفصل يُغطّي الجيل الثاني من تقنيات RAG التي تحل هذه القيود.


## 2. الخريطة الذهنية


```text
RAG المتقدم
├── 1. فهم السؤال
│   ├── إعادة صياغة متعددة (Multi-Query)
│   ├── HyDE: توليد إجابة افتراضية ثم البحث بها
│   └── تفكيك الأسئلة المركّبة إلى أسئلة فرعية
├── 2. الاسترجاع
│   ├── Dense: تشابه متجهات
│   ├── Sparse: BM25 للمطابقة الحرفية
│   ├── Hybrid + Reciprocal Rank Fusion
│   └── فلاتر Metadata: اللغة، التاريخ، الجهة، الصلاحية
├── 3. إعادة الترتيب
│   └── Cross-Encoder يرى السؤال والقطعة معًا
├── 4. بناء السياق
│   ├── ضغط القطع وإزالة التكرار
│   ├── ترتيب المصادر وتحديد الاقتباسات
│   └── التحقق من تعارض الأدلة
├── 5. التوليد
│   ├── إجابة مسنودة بالمصادر (Citations)
│   └── رفض الإجابة عند غياب الدليل
└── 6. التنسيق
    ├── Agentic RAG: استرجاع تكراري بقرار من النموذج
    └── GraphRAG: استرجاع عبر الكيانات والعلاقات
```


## 3. الشرح المتدرج

### 🟢 أ) إعادة الترتيب (Re-ranking) — تحسين سريع وفعّال جدًا

البحث المتجهي الأولي (Bi-encoder) سريع لكن أقل دقة. الحل: استرجاع عدد أكبر (مثلاً 20-50)، ثم **إعادة ترتيبها** بنموذج أدق وأبطأ (Cross-encoder) لاختيار أفضل 5.


In [ ]:
from sentence_transformers import CrossEncoder

class RerankingRAGPipeline(SimpleRAGPipeline):
    def __init__(self, *args, reranker_model="cross-encoder/mmarco-mMiniLMv2-L12-H384-v1", **kwargs):
        super().__init__(*args, **kwargs)
        self.reranker = CrossEncoder(reranker_model)

    def retrieve_with_reranking(self, query: str, initial_k=25, final_k=5) -> list:
        # المرحلة 1: استرجاع واسع وسريع (Bi-encoder)
        candidates = self.retrieve(query, top_k=initial_k)

        # المرحلة 2: إعادة ترتيب دقيقة (Cross-encoder يرى السؤال والمستند معًا)
        pairs = [[query, c["text"]] for c in candidates]
        rerank_scores = self.reranker.predict(pairs)

        for candidate, score in zip(candidates, rerank_scores):
            candidate["rerank_score"] = float(score)

        reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
        return reranked[:final_k]


🟢 **الحدس**: Bi-encoder يُشفِّر السؤال والمستند **بشكل منفصل** ثم يقارن المتجهات (سريع لأن متجهات المستندات محسوبة مسبقًا)، بينما Cross-encoder يُدخل السؤال والمستند **معًا** في نموذج واحد فيرى التفاعل الدقيق بينهما (أدق لكن يجب حسابه لحظيًا لكل زوج، فلا يمكن فهرسته مسبقًا). الجمع بينهما (استرجاع واسع سريع + إعادة ترتيب دقيق محدود) يُحقق أفضل ما في العالمين.

### 🟡 ب) البحث الهجين (Hybrid Search): دلالي + كلمات مفتاحية


In [ ]:
from rank_bm25 import BM25Okapi

class HybridSearchRAG(RerankingRAGPipeline):
    """
    دمج البحث الدلالي (Dense/Embedding) مع البحث اللفظي (Sparse/BM25)
    مهم جدًا للعربية حيث البحث الدلالي قد يفوّت تطابقات حرفية دقيقة
    (أسماء أعلام، مصطلحات تقنية، أرقام)
    """
    def index_documents(self, documents: dict):
        super().index_documents(documents)
        tokenized_chunks = [chunk.split() for chunk in self.chunks]
        self.bm25 = BM25Okapi(tokenized_chunks)

    def hybrid_retrieve(self, query: str, top_k=10, alpha=0.5) -> list:
        """alpha: وزن الموازنة بين البحث الدلالي (1.0) واللفظي (0.0)"""
        # البحث الدلالي
        dense_results = self.retrieve(query, top_k=top_k * 2)
        dense_scores = {r["metadata"]["chunk_index"]: r["score"] for r in dense_results}

        # البحث اللفظي BM25
        bm25_scores_raw = self.bm25.get_scores(query.split())
        # تطبيع BM25 scores إلى [0, 1]
        max_bm25 = max(bm25_scores_raw) if max(bm25_scores_raw) > 0 else 1
        bm25_scores = {i: score / max_bm25 for i, score in enumerate(bm25_scores_raw)}

        # دمج النتائج بالموازنة الخطية (Reciprocal Rank Fusion أو Weighted Sum)
        all_indices = set(dense_scores.keys()) | set(bm25_scores.keys())
        combined_scores = {
            idx: alpha * dense_scores.get(idx, 0) + (1 - alpha) * bm25_scores.get(idx, 0)
            for idx in all_indices
        }

        top_indices = sorted(combined_scores, key=combined_scores.get, reverse=True)[:top_k]
        return [{"text": self.chunks[i], "combined_score": combined_scores[i]} for i in top_indices]


> ⚠️ **خطأ شائع**: الاعتماد فقط على البحث الدلالي (Dense) في التطبيقات التي تتضمن بحثًا عن معرّفات دقيقة (أكواد أخطاء، أرقام قطع غيار، أسماء منتجات) — نماذج embedding تميل لتعميم المعنى، فقد "تُهمل" أو "تُقارب" رقمًا أو معرّفًا فريدًا بدلًا من مطابقته حرفيًا. BM25 يتفوّق تمامًا في هذه الحالات.

### 🔴 ج) طبقة التعمّق: GraphRAG — الاسترجاع عبر رسم بياني للمعرفة

عندما يتطلب السؤال **ربط معلومات متناثرة عبر مستندات متعددة** (مثل: "ما العلاقة بين الشركة X والمشروع Y الذي بدأ بعد استقالة المدير Z؟")، يفشل RAG التقليدي القائم على التشابه المحلي. **GraphRAG** (Microsoft Research) يبني رسمًا بيانيًا للكيانات والعلاقات مسبقًا، ويستخدمه للاستدلال المُركَّب.


In [ ]:
import networkx as nx

class GraphRAGPipeline:
    """
    تطبيق مبسّط لفكرة GraphRAG:
    1. استخراج الكيانات والعلاقات من المستندات (عبر LLM)
    2. بناء رسم بياني للمعرفة
    3. عند الاستعلام: استرجاع عبر المسارات في الرسم، وليس فقط التشابه النصي
    """
    def __init__(self, llm_extractor):
        self.llm_extractor = llm_extractor   # LLM يُستخدم لاستخراج الكيانات/العلاقات
        self.graph = nx.MultiDiGraph()

    def extract_entities_and_relations(self, text_chunk: str) -> list:
        extraction_prompt = f"""استخرج الكيانات (أشخاص، منظمات، أماكن، مفاهيم) والعلاقات بينها
من النص التالي بصيغة JSON: [{{"source": "", "relation": "", "target": ""}}]

النص: {text_chunk}
"""
        response = self.llm_extractor.generate(extraction_prompt)
        import json
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            return []

    def build_graph_from_documents(self, chunks: list):
        for chunk in chunks:
            triples = self.extract_entities_and_relations(chunk)
            for triple in triples:
                self.graph.add_edge(
                    triple["source"], triple["target"],
                    relation=triple["relation"], source_text=chunk
                )

    def community_summarization(self):
        """
        خطوة GraphRAG المحورية: تقسيم الرسم لمجتمعات (communities) مترابطة
        وتلخيص كل مجتمع مسبقًا — يسمح بالإجابة على أسئلة "شاملة" (global queries)
        لا يمكن لـ chunk واحد الإجابة عنها
        """
        undirected = self.graph.to_undirected()
        communities = nx.community.louvain_communities(undirected)

        community_summaries = {}
        for i, community in enumerate(communities):
            subgraph = self.graph.subgraph(community)
            edges_text = "\n".join([
                f"{u} -[{d['relation']}]-> {v}" for u, v, d in subgraph.edges(data=True)
            ])
            summary_prompt = f"لخّص العلاقات التالية في فقرة متماسكة:\n{edges_text}"
            community_summaries[i] = self.llm_extractor.generate(summary_prompt)

        return community_summaries

    def graph_guided_retrieval(self, query_entities: list, max_hops=2) -> list:
        """استرجاع كل العقد المرتبطة بالكيانات المذكورة في السؤال ضمن مسافة معيّنة"""
        relevant_subgraph_nodes = set()
        for entity in query_entities:
            if entity in self.graph:
                neighbors = nx.single_source_shortest_path_length(
                    self.graph.to_undirected(), entity, cutoff=max_hops
                )
                relevant_subgraph_nodes.update(neighbors.keys())

        relevant_edges = [
            (u, v, d) for u, v, d in self.graph.edges(data=True)
            if u in relevant_subgraph_nodes and v in relevant_subgraph_nodes
        ]
        return relevant_edges


### 🟡 د) RAG الوكيلي (Agentic RAG) — تمهيد للفصل القادم

التطور الطبيعي الأخير: بدلاً من استرجاع واحد ثابت، يُقرِّر LLM نفسه **متى وكيف وكم مرة** يبحث:


In [ ]:
def agentic_rag_loop(llm, rag_pipeline, question: str, max_iterations=3):
    """
    حلقة استرجاع تكرارية ذكية: النموذج يقيّم هل المعلومات كافية،
    وإن لم تكن، يُعيد صياغة استعلام بحث جديد ويحاول مجددًا
    """
    accumulated_context = []
    current_query = question

    for iteration in range(max_iterations):
        retrieved = rag_pipeline.retrieve_with_reranking(current_query, final_k=3)
        accumulated_context.extend(retrieved)

        sufficiency_prompt = f"""السؤال الأصلي: {question}
المعلومات المتجمّعة حتى الآن: {[c['text'][:200] for c in accumulated_context]}

هل هذه المعلومات كافية للإجابة الكاملة على السؤال؟
إذا كانت كافية، أجب بـ: "كافٍ"
إذا لم تكن كافية، أجب بـ: "غير كافٍ: [استعلام بحث جديد أكثر تحديدًا]"
"""
        decision = llm.generate(sufficiency_prompt)

        if decision.strip().startswith("كافٍ"):
            break
        elif "غير كافٍ:" in decision:
            current_query = decision.split("غير كافٍ:")[-1].strip()
        else:
            break

    final_prompt = rag_pipeline.build_augmented_prompt(question, accumulated_context)
    return llm.generate(final_prompt)


## 4. مثال محلول خطوة بخطوة

**السؤال:** «ما نسبة نمو مبيعات الشركة X في فرع الرياض بعد تعيين المدير الجديد؟»

لنفترض أن لدينا ثلاثة مستندات مرشّحة، وأن البحث الدلالي والبحث اللفظي أعطيا ترتيبين مختلفين:

| المستند | رتبة Dense | رتبة BM25 | درجة RRF |
|---|---:|---:|---:|
| D1: تقرير مبيعات 2025 | 1 | 3 | 1/61 + 1/63 = 0.0323 |
| D2: قرار تعيين المدير | 2 | 1 | 1/62 + 1/61 = 0.0325 |
| D3: نمو فرع الرياض | 4 | 2 | 1/64 + 1/62 = 0.0317 |

نستخدم Reciprocal Rank Fusion بثابت \(k=60\):

\[
\text{RRF}(d)=\sum_{r \in R}\frac{1}{k+\text{rank}_r(d)}
\]

ثم نمرّر أفضل 20 مرشّحًا إلى Cross-Encoder، فنحصل مثلًا على:

| المستند | درجة Cross-Encoder |
|---|---:|
| D2 | 0.91 |
| D1 | 0.84 |
| D3 | 0.62 |

**النتيجة:** نأخذ D2 وD1 فقط (أعلى درجتين)، ونبني السياق منهما. لاحظ أن D3 كان الأقرب دلاليًا لصياغة السؤال، لكنه لا يحتوي على رقم النمو؛ لذلك لم يُختر. هذا يوضح لماذا لا يكفي التشابه وحده: **الغرض هو العثور على الدليل، لا على النص الأكثر شبهًا بالسؤال**.

**التحقق النهائي:** إذا لم يحتوي السياق على رقم صريح، تُصاغ الإجابة: «لا تتوفر بيانات كافية لحساب النسبة»، بدلًا من تخمين رقم. هذا السلوك (الامتناع عن الإجابة) جزء من جودة RAG وليس فشلًا له.


## 5. نصائح وحيل

> 💡 **طبّع الدرجات بحذر**: درجات BM25 وDense ليست على نفس المقياس. لا تجمعها جمعًا خطيًا مباشرًا قبل التطبيع، وفضّل RRF لأنه يعتمد على الرتب لا على القيم المطلقة.

> 💡 **أعد ترتيب 20-50 مرشّحًا فقط**: Cross-Encoder مكلف؛ تمرير 500 قطعة إليه يقتل زمن الاستجابة دون مكسب يُذكر. جرّب \(k\) مختلفة وقِس nDCG.

> 💡 **أضف فلاتر قبل البحث لا بعده**: إذا كان المستخدم يسأل عن فرع الرياض في 2025، ففلترة التاريخ والموقع تقلّل الضوضاء أكثر من أي تحسين في نموذج التضمين.

> 💡 **خزّن الاقتباسات مع القطعة**: احفظ `doc_id` و`page` و`section` و`language` في الـ metadata؛ ستحتاجها لعرض المصادر، ولتدقيق الإجابات، ولحذف البيانات عند الطلب.

> 💡 **قِس الاسترجاع منفصلًا عن التوليد**: استخدم Recall@k وMRR وnDCG للاسترجاع، وFaithfulness/Answer Relevance للتوليد. إذا كانت الإجابة سيئة، فاعرف أولًا هل المشكلة في الاسترجاع أم في الصياغة.


## 6. أخطاء شائعة ومصائد

> ⚠️ **الاعتماد على Dense فقط للمعرّفات الدقيقة**: أرقام الفواتير، أكواد الأخطاء، أسماء المنتجات، والمصطلحات القانونية قد تُقرَّب دلاليًا أو تُفقد. BM25 ضروري لهذه الحالات.

> ⚠️ **استخدام GraphRAG لمجرد أنه «أحدث»**: بناء الرسم يستدعي LLM لكل قطعة، وقد يكلف أضعاف الفهرسة المتجهية. استخدمه عندما تكون الأسئلة تركيبية فعلًا وتحتاج ربط كيانات متفرقة.

> ⚠️ **تقطيع النص عند حدود عشوائية**: تقطيع كل 500 حرف دون احترام الفقرات أو العناوين يُنتج قطعًا مبتورة. اقطع عند حدود دلالية، واحتفظ بتداخل صغير بين القطع.

> ⚠️ **إغفال تعارض المصادر**: قد يعطي مستندان رقمين مختلفين. يجب أن يتضمن الـ prompt تعليمات صريحة لذكر التعارض والأخذ بالأحدث/الأكثر رسمية، لا لاختيار رقم عشوائيًا.

> ⚠️ **تقييم RAG بسؤالين فقط**: ابنِ مجموعة تقييم عربية تشمل أسئلة محلية، وأسئلة تركيبية، وأسئلة بلا إجابة في القاعدة. السؤال بلا إجابة هو أفضل اختبار لقدرة النظام على الامتناع.


## 7. جدول مقارنة

| المعيار | RAG التقليدي (Vector-based) | GraphRAG |
|---|---|---|
| **أسئلة محلية** (حقيقة واحدة محددة) | ممتاز وسريع | جيد لكن أبطأ |
| **أسئلة شاملة/تركيبية** (ربط عدة مصادر) | ضعيف | ممتاز |
| **تكلفة الفهرسة** | منخفضة | عالية جدًا (استخراج الكيانات عبر LLM لكل قطعة) |
| **قابلية التفسير** | متوسطة | عالية (رسم بياني واضح للعلاقات) |
| **الصيانة عند تحديث البيانات** | سهلة (إضافة/حذف متجهات) | معقّدة (إعادة بناء الرسم جزئيًا) |


## 8. ملخص أفضل الممارسات

- [ ] أسئلة بسيطة محلية فقط → RAG أساسي + Reranking كافٍ
- [ ] أسئلة تتضمن معرّفات/أرقام دقيقة → أضف Hybrid Search (BM25)
- [ ] أسئلة تركيبية تربط عدة مصادر → فكّر في GraphRAG
- [ ] أسئلة غامضة تحتاج توضيحًا متكررًا → Agentic RAG التكراري
- [ ] قيّم دائمًا تكلفة/فائدة التعقيد الإضافي — لا تستخدم GraphRAG إن كان RAG البسيط كافيًا

---


## 9. أسئلة مقابلات

**سؤال 1: لماذا نستخدم Hybrid Search بدلًا من البحث الدلالي وحده؟**

> **إجابة نموذجية:** البحث الدلالي ممتاز في التقاط المعنى والمرادفات، لكنه قد يفشل في المطابقة الحرفية الدقيقة مثل أرقام الفواتير أو أسماء المنتجات. BM25 يتفوق في هذه الحالات لأنه يعتمد على تكرار المصطلحات وتطابقها. الدمج بينهما عبر RRF يعطي تغطية أوسع: الأول يلتقط «المعنى»، والثاني يلتقط «اللفظ». في العربية يزداد الأمر أهمية بسبب تعدد الصيغ واللهجات، فبعض المصطلحات التقنية تبقى كما هي بالإنجليزية ويحتاج النظام لمطابقتها حرفيًا.

**سؤال 2: متى تختار GraphRAG على RAG المتجهي؟**

> **إجابة نموذجية:** عندما تكون الأسئلة «شاملة» أو تركيبية وتتطلب ربط معلومات من عدة مستندات، مثل «ما العلاقة بين تعيين المدير الجديد وانخفاض معدل الدوران في الفرع؟». RAG المتجهي يسترجع قطعًا متشابهة محليًا، لكنه لا يملك تمثيلًا صريحًا للعلاقات. GraphRAG يبني كيانات وعلاقات ومجتمعات دلالية مسبقًا، فيستطيع السير عبر المسارات وتلخيص المجتمعات. الثمن: تكلفة فهرسة عالية وصيانة معقدة، لذلك لا يبرَّر إلا إذا كان نوع الأسئلة يستحق.

**سؤال 3: كيف تقيّم نظام RAG بشكل عملي؟**

> **إجابة نموذجية:** أفصل التقييم إلى طبقتين. طبقة الاسترجاع: Recall@k وMRR وnDCG على مجموعة أسئلة مع مستندات ذهبية. طبقة التوليد: Faithfulness (هل كل ادعاء مسنود؟)، Answer Relevance، وCitation Accuracy. أضيف مجموعة أسئلة «غير قابلة للإجابة» لقياس الامتناع، وأتابع تكلفة وزمن الاستجابة. التقييم يجب أن يكون آليًا في CI مع عينة بشرية دورية، لا تقييمًا يدويًا لمرة واحدة.


## 10. تمرين عملي

**الهدف:** بناء خط RAG عربي صغير ومقارنة ثلاث نسخ: Dense فقط، Hybrid، ثم Hybrid + Reranker.

**الخطوات:**
1. اجمع 30 مستندًا عربيًا قصيرًا (أخبار، وثائق تقنية، أو صفحات دعم) و30 سؤالًا، منها 5 أسئلة بلا إجابة في المجموعة.
2. قسّم المستندات إلى قطع دلالية، واحفظ `doc_id` و`section` و`date` في الـ metadata.
3. ابنِ فهرسين: متجهي (باستخدام `sentence-transformers` أو أي API تضمين) وBM25.
4. نفّذ الدمج بـ RRF، ثم أضف Cross-Encoder لإعادة ترتيب أفضل 20.
5. احسب Recall@5 وMRR لكل نسخة، وسجّل زمن الاستجابة لكل سؤال.
6. أضف 5 أسئلة مركّبة تحتاج ربط مستندين، وقارن هل تحسّنت الإجابات أم لا.
7. اكتب تقريرًا من صفحة واحدة: أي نسخة أفضل، وبأي تكلفة، وما الحالات التي فشلت فيها.

**معيار النجاح:** تحسّن Recall@5 بنسبة 10% على الأقل عند الانتقال من Dense إلى Hybrid، دون مضاعفة زمن الاستجابة أكثر من مرتين.


## 11. مراجع للتعمّق

- Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS.
- Karpukhin, V., et al. (2020). *Dense Passage Retrieval for Open-Domain Question Answering*. EMNLP.
- Cormack, G. V., Clarke, C. L. A., & Buettcher, S. (2009). *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods*. SIGIR.
- Edge, D., et al. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused Summarization*. Microsoft Research.
- Gao, Y., et al. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey*. arXiv:2312.10997.
- Nogueira, R., & Cho, K. (2019). *Passage Re-ranking with BERT*. arXiv:1901.04085.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التطبيقات والوكلاء الذكية / الفصل 48: تقنيات RAG المتقدمة وGraphRAG.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
